# Capa 4 — Análisis Clásico (Estadística Inferencial)
## Proyecto: Customer Analytics — "Super Compras"

**Metodología:** 6 Capas (Estrategia → Preparación → EDA → **Análisis Clásico** → ML → Entregable)

**Nivel de la pirámide:** Nivel II (Diagnóstico/Inferencial). Este notebook contrasta con pruebas
formales (p<0,05) los patrones que la Capa 3 solo pudo mostrar de forma visual/descriptiva.

**Punto de partida:** las 5 preguntas cerradas al final de `03_capa3_eda.ipynb`, más los
contrastes adicionales de Segmento/sociodemográficas necesarios para cerrar la Capa 1
(Decisión de negocio: priorizar presupuesto según valor real de cada segmento).

**Regla de oro de este notebook (checklist del proyecto):** ningún contraste se interpreta sin
antes (1) verificar sus supuestos, (2) reportar el tamaño de efecto — no solo el p-valor — y
(3) corregir por comparaciones múltiples cuando se lanzan varios tests a la vez.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from itertools import combinations

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 100
RUTA = "./"   # carpeta donde está cliente_nivel_eda.csv


In [ ]:
cli = pd.read_csv(RUTA + "cliente_nivel_eda.csv")
cli["En_Riesgo"] = (cli["Segmento"] == "En Riesgo").astype(int)
print("Clientes:", cli.shape[0])
cli[["Recency", "Frequency", "Monetary", "ltv", "CAC", "Segmento", "Genero",
     "Nivel_Educativo", "Ocupacion", "Tamano_Hogar", "En_Riesgo"]].head()


## Paso previo — Verificación de supuestos

Antes de elegir t-Student/ANOVA (paramétricos) o Mann-Whitney/Kruskal-Wallis (no paramétricos),
se comprueban formalmente normalidad (Kolmogorov-Smirnov, H0: los datos son normales) y
homocedasticidad (Levene, H0: las varianzas son iguales entre grupos).

In [ ]:
print("--- Normalidad (Kolmogorov-Smirnov) ---")
for v in ["Monetary", "ltv", "Frequency", "Recency", "CAC"]:
    stat, p = stats.kstest(cli[v], "norm", args=(cli[v].mean(), cli[v].std()))
    print(f"{v:10s} p={p:.4g} -> {'Normal' if p > 0.05 else 'NO normal'}")

print("\n--- Homocedasticidad (Levene) sobre Monetary ---")
for cat in ["Genero", "Tamano_Hogar", "Nivel_Educativo", "Ocupacion"]:
    groups = [g["Monetary"].values for _, g in cli.groupby(cat)]
    stat, p = stats.levene(*groups)
    print(f"Monetary ~ {cat:16s} p={p:.4g} -> {'Varianzas iguales' if p > 0.05 else 'Varianzas DISTINTAS'}")


**Decisión:** ninguna variable cuantitativa es normal (p≈0 en las 5) y tampoco hay
homocedasticidad en ninguno de los cruces (p<0,0001 en los 4). **Se descartan t-Student, Welch y
ANOVA** — todos los contrastes de esta capa usan la rama no paramétrica: **Mann-Whitney U** (2
grupos) y **Kruskal-Wallis** (3+ grupos), con post-hoc por pares (Mann-Whitney + corrección FDR)
cuando el Kruskal-Wallis resulte significativo.

## Bloque 1 — ¿Monetary difiere por variables sociodemográficas?

### 1.1 — Genero (2 grupos) → Mann-Whitney U

In [ ]:
h = cli[cli.Genero == "Hombre"]["Monetary"]
m = cli[cli.Genero == "Mujer"]["Monetary"]
u_stat, p = stats.mannwhitneyu(h, m, alternative="two-sided")
rank_biserial = 1 - (2 * u_stat) / (len(h) * len(m))

print(f"Mediana Monetary Hombre = {h.median():.2f} EUR (n={len(h)})")
print(f"Mediana Monetary Mujer  = {m.median():.2f} EUR (n={len(m)})")
print(f"U = {u_stat:.1f}   p-valor = {p:.4g}")
print(f"Tamaño de efecto (r biserial por rangos) = {rank_biserial:.4f}  (<0,1 = negligible)")


**Lectura:** la diferencia es estadísticamente significativa (p<0,001), pero el tamaño de
efecto (r=0,035) es **negligible**. Con n=14.367 hasta una diferencia de 3,4€ en la mediana
(68,46€ vs 71,84€) resulta "significativa" — significancia estadística no es lo mismo que
relevancia de negocio. **Género no es un criterio útil para diferenciar estrategias de
marketing por gasto.**

### 1.2 — Tamano_Hogar, Nivel_Educativo, Ocupacion (3+ grupos) → Kruskal-Wallis

In [ ]:
def kruskal_eta2(df, cat, val="Monetary"):
    groups = [g[val].values for _, g in df.groupby(cat)]
    H, p = stats.kruskal(*groups)
    k, n = len(groups), len(df)
    eta2 = (H - k + 1) / (n - k)   # tamaño de efecto epsilon^2 aproximado
    return H, p, eta2

resultados_kw = []
for cat in ["Tamano_Hogar", "Nivel_Educativo", "Ocupacion"]:
    H, p, eta2 = kruskal_eta2(cli, cat)
    mag = "negligible" if eta2 < 0.01 else ("pequeño" if eta2 < 0.06 else ("mediano" if eta2 < 0.14 else "grande"))
    resultados_kw.append([cat, H, p, eta2, mag])
    print(f"{cat:18s} H={H:8.2f}  p={p:.4g}  eta2={eta2:.4f} ({mag})")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, cat in zip(axes, ["Tamano_Hogar", "Nivel_Educativo", "Ocupacion"]):
    order = cli.groupby(cat)["Monetary"].median().sort_values(ascending=False).index
    sns.boxplot(data=cli, x=cat, y="Monetary", order=order, ax=ax, color="#66BB6A", showfliers=False)
    ax.set_title(cat)
    ax.tick_params(axis="x", rotation=35)
plt.tight_layout()
plt.show()


**Lectura:**
- `Tamano_Hogar`: significativo (p=0,0095) pero eta²=0,0005 — **negligible**, no aporta nada
  en la práctica.
- `Nivel_Educativo` (p=6·10⁻⁵⁵) y `Ocupacion` (p=7·10⁻⁶⁹): significativos **y** con efecto
  "pequeño" (eta²≈0,018-0,0225, 1,8%-2,3% de la varianza de Monetary). Son los dos únicos
  cruces sociodemográficos de este bloque con relevancia real, aunque modesta.

### 1.3 — Post-hoc (Mann-Whitney por pares + corrección FDR)

Solo tiene sentido hacer post-hoc en los 3 Kruskal-Wallis, ya que los 3 dieron significativos.
La corrección FDR se aplica DENTRO de cada variable (familia de comparaciones por pares de esa
variable), no mezclada con las otras.

In [ ]:
def posthoc_mannwhitney(df, cat, val="Monetary"):
    cats = sorted(df[cat].unique())
    rows = []
    for a, b in combinations(cats, 2):
        xa, xb = df[df[cat] == a][val], df[df[cat] == b][val]
        stat, p = stats.mannwhitneyu(xa, xb, alternative="two-sided")
        rows.append({"grupo_a": a, "grupo_b": b, "mediana_a": xa.median(), "mediana_b": xb.median(), "p_raw": p})
    out = pd.DataFrame(rows)
    out["p_fdr"] = stats.false_discovery_control(out["p_raw"])
    out["sig_fdr_0.05"] = out["p_fdr"] < 0.05
    return out.sort_values("p_raw").reset_index(drop=True)

ph_hogar = posthoc_mannwhitney(cli, "Tamano_Hogar")
ph_educ = posthoc_mannwhitney(cli, "Nivel_Educativo")
ph_ocup = posthoc_mannwhitney(cli, "Ocupacion")

print("Tamano_Hogar: pares significativos tras FDR:", ph_hogar["sig_fdr_0.05"].sum(), "de", len(ph_hogar))
print("Nivel_Educativo: pares significativos tras FDR:", ph_educ["sig_fdr_0.05"].sum(), "de", len(ph_educ))
print("Ocupacion: pares significativos tras FDR:", ph_ocup["sig_fdr_0.05"].sum(), "de", len(ph_ocup))
ph_educ


**Lectura del post-hoc de `Nivel_Educativo`** (12 de 15 pares significativos tras FDR): el
patrón es consistente — **Máster (mediana 88,47€) y Universitario (79,57€) gastan
significativamente más que Educación básica (56,96€), Sin estudios (55,48€) y Desconocido
(58,20€).** Los únicos 3 pares NO significativos son, precisamente, las comparaciones dentro del
grupo de menor gasto (Desconocido vs Educación básica, Desconocido vs Sin estudios, Educación
básica vs Sin estudios) — que se comportan como un bloque homogéneo entre sí, distinto del
bloque Máster/Universitario/Bachillerato.

El patrón en `Ocupacion` es análogo: Autónomo (88,96€) y Empleado/Jubilado (~75-76€) gastan más
que Desempleado/Estudiante/Desconocido (~55€), con 11 de 15 pares significativos tras FDR.

## Bloque 2 — Asociación entre variables categóricas (Chi-cuadrado + Cramér's V)

**Por qué se añade Cramér's V y no solo el p-valor:** con n=14.367, el chi-cuadrado tiene
muchísima potencia estadística — puede marcar como "significativa" una asociación real pero
minúscula. Cramér's V (0 a 1) mide la magnitud, independiente del tamaño de muestra:
<0,1 negligible · 0,1-0,3 débil · 0,3-0,5 moderado · >0,5 fuerte.

In [ ]:
def chi2_cramer(df, c1, c2):
    ct = pd.crosstab(df[c1], df[c2])
    chi2, p, dof, expected = stats.chi2_contingency(ct)
    n, k = ct.values.sum(), min(ct.shape)
    cramer_v = np.sqrt(chi2 / (n * (k - 1)))
    return chi2, p, dof, cramer_v, expected.min()

pares = [("Segmento", "Genero"), ("En_Riesgo", "Genero"), ("En_Riesgo", "Nivel_Educativo"),
         ("En_Riesgo", "Ocupacion"), ("En_Riesgo", "Tamano_Hogar")]

filas = []
for c1, c2 in pares:
    chi2, p, dof, v, min_exp = chi2_cramer(cli, c1, c2)
    mag = "negligible" if v < 0.1 else ("débil" if v < 0.3 else ("moderado" if v < 0.5 else "fuerte"))
    filas.append([f"{c1} x {c2}", chi2, dof, p, v, mag, min_exp])
    assert min_exp >= 5, f"Supuesto de chi-cuadrado (frecuencia esperada >=5) violado en {c1}x{c2}"

chi2_df = pd.DataFrame(filas, columns=["Cruce", "chi2", "gl", "p_raw", "CramerV", "magnitud", "min_frec_esperada"])
chi2_df


**Lectura:** de los 5 cruces, `Segmento x Genero` (p=0,116) y `En_Riesgo x Genero` (p=0,054)
NO son significativos — el género no distingue el riesgo de abandono. Los otros 3
(`Nivel_Educativo`, `Ocupacion`, `Tamano_Hogar` frente a `En_Riesgo`) sí dan p<0,05, pero con
Cramér's V entre 0,024 y 0,089 — **todos "negligible"**. Conclusión honesta: **ninguna variable
sociodemográfica disponible predice de forma relevante quién cae en el segmento "En Riesgo".**
Esto es un hallazgo importante de cara a la Capa 5: si se necesita anticipar el riesgo de
abandono, va a hacer falta señal de **comportamiento temprano** (M1), no perfil sociodemográfico.

## Bloque 3 — Revisión de un hallazgo de la Capa 3: correlación de CAC

En la Capa 3 se calculó la correlación de **Pearson** entre CAC y Monetary/ltv (r≈0,13-0,14) y
se concluyó que "CAC prácticamente no correlaciona con nada". **Esa lectura estaba incompleta:**
Pearson mide relación *lineal* y ya sabemos (Capa 3) que estas variables no son normales y están
muy sesgadas — Pearson subestima la asociación real en ese escenario. La medida correcta aquí es
**Spearman** (correlación de rangos, no exige linealidad ni normalidad).

In [ ]:
for v in ["Monetary", "ltv"]:
    rho, p = stats.spearmanr(cli["CAC"], cli[v])
    mag = "negligible" if abs(rho) < 0.1 else ("débil" if abs(rho) < 0.3 else ("moderado" if abs(rho) < 0.5 else "fuerte"))
    print(f"Spearman CAC ~ {v:10s} rho={rho:.4f}  p={p:.4g}  ({mag})")


**Corrección explícita del hallazgo de la Capa 3:** con Spearman, CAC sí tiene una
correlación real con Monetary y ltv (rho≈0,23, p≈0) — **débil, pero ya no negligible.** La
lectura correcta no es "el CAC no predice nada", sino: **el CAC tiene una relación monótona
positiva débil con el valor futuro del cliente — apunta en la dirección correcta, pero es
insuficiente por sí solo** para decidir el presupuesto de adquisición. Se mantiene la
recomendación de fondo (no usar solo CAC para decidir inversión en captación), pero con la
magnitud correcta.

## Bloque 4 — Corrección global por comparaciones múltiples

Se han lanzado 11 contrastes "omnibus" en esta capa (1 Mann-Whitney + 3 Kruskal-Wallis + 5
Chi-cuadrado + 2 Spearman). Aplicar FDR (Benjamini-Hochberg) sobre el conjunto completo evita
inflar el error de tipo I por probar varias hipótesis a la vez — es la tabla resumen de toda la
Capa 4.

In [ ]:
tests = []

u_stat, p = stats.mannwhitneyu(h, m)
tests.append(["Mann-Whitney: Monetary ~ Genero", p, f"r={rank_biserial:.3f}", "negligible"])

for cat, (H, p, eta2) in zip(["Tamano_Hogar", "Nivel_Educativo", "Ocupacion"],
                               [kruskal_eta2(cli, c) for c in ["Tamano_Hogar", "Nivel_Educativo", "Ocupacion"]]):
    mag = "negligible" if eta2 < 0.01 else ("pequeño" if eta2 < 0.06 else "mediano")
    tests.append([f"Kruskal-Wallis: Monetary ~ {cat}", p, f"eta2={eta2:.4f}", mag])

for _, row in chi2_df.iterrows():
    tests.append([f"Chi-cuadrado: {row['Cruce']}", row["p_raw"], f"CramerV={row['CramerV']:.4f}", row["magnitud"]])

for v in ["Monetary", "ltv"]:
    rho, p = stats.spearmanr(cli["CAC"], cli[v])
    mag = "negligible" if abs(rho) < 0.1 else "débil"
    tests.append([f"Spearman: CAC ~ {v}", p, f"rho={rho:.4f}", mag])

resumen = pd.DataFrame(tests, columns=["Test", "p_raw", "efecto", "magnitud"])
resumen["p_fdr"] = stats.false_discovery_control(resumen["p_raw"])
resumen["sig_fdr_0.05"] = resumen["p_fdr"] < 0.05
resumen.to_csv("resumen_omnibus_capa4.csv", index=False)
resumen


**Balance final de la Capa 4:** 9 de los 11 contrastes siguen siendo significativos tras la
corrección FDR — pero solo **3 tienen un tamaño de efecto por encima de "negligible"**:
`Nivel_Educativo` y `Ocupacion` sobre Monetary (efecto "pequeño") y `CAC` sobre Monetary/ltv
(efecto "débil"). Con una muestra de 14.367 clientes, el p-valor por sí solo es un mal filtro —
la magnitud del efecto es la que separa lo accionable de lo anecdótico.

## Resumen ejecutivo de la Capa 4 (Nivel II cerrado)

| Pregunta (de la Capa 3) | Resultado | ¿Accionable? |
|---|---|---|
| ¿"En Riesgo" se explica por sociodemográficas? | Género: no significativo. Nivel_Educativo/Ocupacion/Tamano_Hogar: significativos pero Cramér's V negligible (<0,09) | **No** — hace falta señal de comportamiento, no de perfil |
| ¿La correlación CAC-Monetary es real? | Sí, con Spearman (rho≈0,23, débil) — el análisis de Pearson de la Capa 3 la había subestimado | Débil; insuficiente en solitario |
| ¿Segmento y Género son independientes? | Sí, independientes (p=0,116) | — |
| ¿Monetary/ltv son normales? | Confirmado: no lo son (K-S, todas las variables) | Se usó la rama no paramétrica en toda la capa |
| ¿Diferencia de Monetary por Nivel_Educativo es real? | Sí, significativa y con efecto pequeño-real (eta²=0,018); Máster/Universitario gastan más que Educación básica/Sin estudios/Desconocido | **Sí** — segmentar comunicación por nivel educativo tiene base estadística |

**Conclusión de negocio para la Capa 6:** los datos sociodemográficos disponibles (género,
tamaño de hogar) NO son buena base para diferenciar campañas — los efectos son estadísticamente
detectables pero irrelevantes en magnitud. Nivel educativo y ocupación sí aportan una señal
pequeña pero real. Ninguna variable sociodemográfica predice de forma útil el riesgo de
abandono ("En Riesgo") — la Capa 5 debe apoyarse en comportamiento temprano (M1), no en perfil.

**Siguiente paso:** `05_capa5_prediccion_churn.ipynb` — modelo de riesgo de abandono temprano
(activo en M3/M6), usando señales de M1 + sociodemográficas como predictores exploratorios
(sabiendo, por este mismo notebook, que su aporte individual es débil) y evitando
`Monetary`/`RFM`/`Segmento` como variables de entrada por circularidad.